In [16]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random
from urllib.parse import urlparse, parse_qs

In [17]:
# Get a list of every event for a season

season = "2026" # Last year of the season
category = "mens"

filtertype = {
    "mens": 81,
    "womens": 82
}

data = {
    "name": [],
    "id": []
}


url = f'https://home.curlingzone.com/schedule.php?filtertype={filtertype[category]}&eventyear={season}'
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Loop through each event element
for el in soup.select(".featured-title"):
    link = el.find_all('a')[0].get('href') # get the anchor tag with the link to the event
    data['name'].append(el.find_all('a')[0].text)
    data['id'].append(link[link.index("eventid=")+8:]) # parse out the event id
    

events = pd.DataFrame(data)
events.to_csv(f"data/{category}/events.csv", index=False)
events



,name,id
0,Morioka City Mens Memorial Cup (Men),9106
1,Australia Mens Curling Championships (Men),9078
2,Korean National Curling Championship (Men),9206
3,Wakkanai Midori Challenge Cup (Men),9064
4,Hokkaido Bank Curling Classic (Men),9067
...,...,...
222,Uiseong Governor's Cup (Men),9581
223,World Senior Curling Championships (Men),9253
224,U25 NextGen Classic (Men),9449
225,World Championship Pre-Qualifier (Europe) (Men),9579


In [18]:
# Gather list of teams in a particular season

data = {
    "id": [],
    "name": [],
    "location": [],
    "lead": [],
    "second": [],
    "third": [],
    "fourth": [],
    "alternate": [],
    "coach": [],
}

url = f"https://home.curlingzone.com/teams.php?et={filtertype[category]}&ey={season}"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Select each team element
for el in soup.select(".team-card"):
    name = el.find('div').find('a').text
    id = el.find('div').find('a').get('href')
    id = id[id.index("teamid=")+7:id.index("teamid=")+13]
    infobox = el.select('.team-card-body')[0]
    lead = None
    second = None
    third = None
    fourth = None
    alternate = None
    coach = None

    try:
        location = infobox.select(".team-location")[0].text
    except:
        location = None

    try:
        coach = infobox.select(".team-coach")[0].text
        coach = coach[coach.index("Coach: ")+7:]
    except:
        coach = None

    # Go through each player row, and assign players to their respective position.
    # (Not overly concerned about leads on 3-player teams being listed as seconds.)
    for index, player in enumerate(infobox.select(".team-pos-row")):
        if index == 0:
            fourth = player.select(".team-pos-name")[0].text
        elif index == 1:
            third = player.select(".team-pos-name")[0].text
        elif index == 2:
            second = player.select(".team-pos-name")[0].text
        elif index == 3:
            lead = player.select(".team-pos-name")[0].text
        elif index == 4:
            alternate = player.select(".team-pos-name")[0].text

    data['id'].append(id)
    data['name'].append(name)
    data['location'].append(location)
    data['lead'].append(lead)
    data['second'].append(second)
    data['third'].append(third)
    data['fourth'].append(fourth)
    data['alternate'].append(alternate)
    data['coach'].append(coach)

print(data)
teams = pd.DataFrame(data)
teams.to_csv(f"data/{category}/teams.csv", index=False)
    

{'id': ['191992', '100498', '100006', '100090', '192079', '100354', '100454', '100109', '100114', '100342', '100353', '100048', '100056', '100009', '100234', '100038', '100495', '191785', '192693', '100500', '100278', '100028', '100028', '100106', '100398', '100106', '100353', '191836', '100099', '100280', '100099', '100335', '100205', '100057', '194730', '100098', '100402', '100387', '100106', '194731', '100234', '100406', '100009', '193642', '192706', '100156', '100342', '194174', '100308', '194988', '192338', '100154', '100385', '100055', '192583', '192290', '195365', '100205', '100442', '100009', '100358', '195358', '194803', '100353', '100442', '192392', '192463', '100006', '100006', '192347', '193247', '100278', '100352', '100191', '100191', '100090', '100130', '100305', '192835', '100232', '195272', '193633', '100125', '100402', '192422', '192311', '100007', '100488', '100344', '192088', '192415', '100278', '100090', '195359', '100234', '100245', '192983', '194732', '100336', '1

In [ ]:
# FOR EACH EVENT
# Go to the teams page
# Look at each team and go to the "team profile" hyperlink to gather their id
# temporarily keep the event specific team id and global id in a dictionary
#  "localid": globalid
# when looking at each individual game, reference that id so everything is the same.
# when looking at a game, we unfortunately have to go to the actual team page. :( (gerry will hate me)




data = {
    "id": [],
    "event": [],
    "draw":[],
    "teamA": [],
    "teamB": [],
    "lsfe": [],
    "endsScheduled": [],
    "scorelineA": [],
    "scorelineB": [],
    "scoreA":[],
    "scoreB":[],
    "winner":[],
}

debug = {
    "id": [],
    "event": [],
    "draw":[],
    "err": []
}

# Iterate through every event
for _, event in events["id"].items():
    # Team ID Gathering:
    teams = {}
    

    url = f"https://home.curlingzone.com/event.php?view=Teams&eventid={event}"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    division = soup.select("#et-pair-tabs")[0].find("li").find("a").text

    # Iterate through each team in the event
    for _, team_card in enumerate(soup.select(".et-name-link")):
        local_id = parse_qs(urlparse(team_card.get("href")).query).get("teamid")[0]

        # Visit their team page to get the global id
        url = team_card.get("href")
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")

        global_id_link = urlparse(soup.select(".alert")[0].find_all('a')[1].get('href'))
        global_id = parse_qs(global_id_link.query).get("teamid")[0]

        teams[local_id] = global_id # add ids to the teams dict
        time.sleep(random.randint(1, 16))


    # Game Information Gathering
    url = f"https://home.curlingzone.com/event.php?view=Scores&eventid={event}"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    draws = []
    draws_dropdown = soup.select('select[name="showdrawid"]')[0]

    for draw in draws_dropdown.find_all("option"):
        draws.append(draw.get("value"))

    #Fetch the specific page for each draw
    for draw in draws:
        url = f"https://home.curlingzone.com/event.php?view=Scores&eventid={event}&showdrawid={draw}"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")

        # Find every game link for that draw, and fetch the page
        for game_box in soup.select(".card"):
            # Some events with both a men's and womens events will lump scores together
            # Double check that the division is the same before analyzing the game.
            if len(game_box.select(".badge.bg-info.text-dark")) !=0:
                if game_box.select(".badge.bg-info.text-dark")[0].text != division:
                    print(division, event, draw)
                    continue

            for game_link in game_box.select(".btn.btn-xs.btn-outline-secondary.ms-auto.py-0"):
                url = game_link.get('href')
                response = requests.get(url)
                soup = BeautifulSoup(response.text, "html.parser")

                game_summary_box = soup.select(".card-body")[0]
                linescore_box = soup.select(".linescore-table")[0]

                #Start parsing all the data about this game
                id = parse_qs(urlparse(url).query).get("showgameid")[0]
                
                # In case some data doesnt work
                try:

                    teamA_url = game_summary_box.select(".gm-team-link")[0].get('href')
                    teamB_url = game_summary_box.select(".gm-team-link")[1].get('href')
                    teamA = teams[parse_qs(urlparse(teamA_url).query).get('teamid')[0]]
                    teamB = teams[parse_qs(urlparse(teamB_url).query).get('teamid')[0]]

                    winner = None
                    if "gm-winner" in game_summary_box.select(".gm-team-row")[0].get("class", []):
                        winner = teamA
                    elif "gm-winner" in game_summary_box.select(".gm-team-row")[1].get("class", []):
                        winner = teamB

                    teamA_linescore_box = linescore_box.find("tbody").find("tr")
                    teamB_linescore_box = linescore_box.find("tbody").find_all("tr")[1]

                    endsScheduled = len(teamA_linescore_box.select(".ls-end:not(.ls-ee)"))
                    scoreA = teamA_linescore_box.select(".ls-total")[0].text
                    scoreB = teamB_linescore_box.select(".ls-total")[0].text

                    scorelineA = ""
                    scorelineB = ""

                    # Iterate through each end box and add it to the scoreline string
                    for team in [teamA_linescore_box, teamB_linescore_box]:
                        for _, end in enumerate(team.select(".ls-end")):
                            if team == teamA_linescore_box:
                                scorelineA += end.text
                            else:
                                scorelineB += end.text

                    lsfe = teamA if teamA_linescore_box.select(".ls-hmr-col")[0].find('span') != None else teamB

                    #Assemble data for dataframe
                    data['event'].append(event)
                    data['id'].append(id)
                    data['draw'].append(draw)
                    data['teamA'].append(teamA) 
                    data['teamB'].append(teamB) 
                    data['lsfe'].append(lsfe) 
                    data['endsScheduled'].append(endsScheduled) 
                    data['scorelineA'].append(scorelineA) 
                    data['scorelineB'].append(scorelineB) 
                    data['scoreA'].append(scoreA) 
                    data['scoreB'].append(scoreB) 
                    data['winner'].append(winner) 

                    # Cool down to reduce the risk of rate limiting
                    time.sleep(random.randint(1, 16))

                except Exception as e:
                    # If a game is broken, record the data we had.
                    debug['event'].append(event)
                    debug['id'].append(id)
                    debug['draw'].append(draw)
                    debug['err'].append(e)

                    broken_games = pd.DataFrame(debug)
                    broken_games.to_csv("debug.csv")

        games = pd.DataFrame(data)
        games.to_csv(f'data/{category}/games.csv')
        time.sleep(random.randint(1, 16))








Men 9078 2
Men 9078 3
Men 9078 3
Men 9078 4
Men 9078 4
Men 9078 5
Men 9078 5
Men 9078 6
Men 9078 6
Men 9078 7
Men 9078 8
Men 9078 8
Men 9078 9
Men 9078 10
Men 9078 11
Men 9078 12
Men 9206 19
Men 9206 19
Men 9206 20
Men 9064 11
Men 9064 11
Men 9064 12
Men 9064 13
Men 9067 1
Men 9067 1
Men 9067 2
Men 9067 2
Men 9067 3
Men 9067 3
Men 9067 4
Men 9067 4
Men 9067 5
Men 9067 5
Men 9067 6
Men 9067 6
Men 9067 7
Men 9067 7
Men 9067 8
Men 9067 8
Men 9067 9
Men 9067 9
Men 9067 10
Men 9067 10
Men 9067 11
Men 9067 11
Men 9067 12
Men 9067 13
Men 9136 1
Men 9136 2
Men 9136 3
Men 9136 4
Men 9136 5
Men 9136 6


In [ ]:
teams

{'191781': '191785',
 '191779': '191786',
 '191780': '191787',
 '191782': '191788',
 '191783': '191790',
 '191784': '191789'}